In [ ]:
using CSV
using DataFrames
using GLM
using Plots
using Random
using Statistics

url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
data = CSV.read(download(url), DataFrame)

In [ ]:
# Анализ данных
println("\nРазмерность данных: ", size(data))

println("Первые 10 строк данных:")
display(first(data, 10))

println("\nСтатистика данных:")
display(describe(data))

histogram(data.medv, xlabel="Цена (medv)", ylabel="Частота", title="Распределение цен")

In [ ]:
# Разделение данных на обучающую и тестовую выборки
Random.seed!(777)
n = nrow(data)
shuffledIndexes = shuffle(1:n) 
trainSize = Int(round(0.8 * n)) 
trainIndexes = shuffledIndexes[1:trainSize]
testIndexes = shuffledIndexes[trainSize+1:end]
trainDataFrame = data[trainIndexes, :]
testDataFrame = data[testIndexes, :]

In [ ]:
# Построение модели линейной регрессии со всеми признаками
formula = @formula(medv ~ crim + zn + indus + chas + nox + rm + age + dis + rad + tax + ptratio + b + lstat) 
model = lm(formula, trainDataFrame) 

# Вывод коэффициентов модели
println("\nКоэффициенты модели:")
ct = coeftable(model)
for (name, coef) in zip(coefnames(model), coef(model)) 
    println(rpad(name, 12), " = ", round(coef, digits=4))
end

In [ ]:
# Прогнозирование и оценка модели
y_true = testDataFrame.medv
y_pred = predict(model, testDataFrame)

# Различные метрики оценки
mse = mean((y_true .- y_pred).^2)
rmse = sqrt(mse) 
mae = mean(abs.(y_true .- y_pred))
r2 = 1 - sum((y_true .- y_pred).^2) / sum((y_true .- mean(y_true)).^2) 

println("\nОценка модели на тестовой выборке:")
println("Среднеквадратичная ошибка (MSE): ", round(mse, digits=2))
println("Среднеквадратичная ошибка (RMSE): ", round(rmse, digits=2))
println("Средняя абсолютная ошибка (MAE): ", round(mae, digits=2))
println("Коэффициент детерминации (R²): ", round(r2, digits=2))

In [ ]:
# Визуализация предсказаний vs реальных значений
scatter(y_true, y_pred, xlabel="Реальные значения", ylabel="Предсказанные значения", 
        title="Предсказанные vs Реальные значения", legend=false)
plot!(LinRange(minimum(y_true), maximum(y_true), 10), 
      LinRange(minimum(y_true), maximum(y_true), 10), 
      linewidth=3, linecolor=:red)